In [5]:
!git clone https://github.com/alps-lab/remasker.git
%cd remasker

Cloning into 'remasker'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 48 (delta 7), reused 41 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 1.08 MiB | 11.60 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/remasker


In [6]:
!pip install timm hyperimpute

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.2 MB/s eta 0:00:00


In [7]:
!sed -i 's/dtype=np\.float)/dtype=float)/' utils.py
!grep -n "np\.float" utils.py

In [8]:
src = open("plugin_mae.py").read()

start = src.index("        cka = {} \n")
end = src.index("        self.model.train()\n", start)
src = src[:start] + src[end:]

start = src.index("        miss_rates = [0.1, 0.3, 0.5, 0.7]\n")
end = src.index("        for epoch in range(self.max_epochs):", start)
src = src[:start] + "\n" + src[end:]

start = src.index("            if (epoch + 1) % 10 == 0:\n                # check CKA")
end = src.index("            # if total_loss < best_loss:", start)
src = src[:start] + src[end:]

start = src.index("        with open('/data/ting/remasker2/output/letter-cka'")
end = src.index("        # torch.save(self.model.state_dict(), self.path)\n        return self", start)
src = src[:start] + src[end:]

open("plugin_mae.py", "w").write(src)
print("patched")

patched


In [9]:
!pwd
!nvidia-smi
!grep -n "ting/remasker2\|letter.pt" plugin_mae.py

/content/remasker
Wed Oct 21 08:46:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------

In [10]:
!sed -i "s/^methods = .*/methods = ['ice', 'mice', 'missforest', 'mean', 'median', 'most_frequent']/" run_baseline.py
!grep -n "^methods" run_baseline.py

27:methods = ['ice', 'mice', 'missforest', 'mean', 'median', 'most_frequent']


In [12]:
%%time
!python run_baseline.py --dataset yacht --exp_name baselines --note run1

RMSE score
  Scenario  miss_pct [0, 1]  ...             median      most_frequent
0      MAR              0.3  ...  0.2965 +/- 0.0206  0.3296 +/- 0.0359

[1 rows x 9 columns]


Wasserstein score
  Scenario  miss_pct [0, 1]  ...            median     most_frequent
0      MAR              0.3  ...  0.258 +/- 0.0219  0.2821 +/- 0.029

[1 rows x 9 columns]


AUROC score
  Scenario  miss_pct [0, 1]  ...       median most_frequent
0      MAR              0.3  ...  0.0 +/- 0.0   0.0 +/- 0.0

[1 rows x 9 columns]
CPU times: user 212 ms, sys: 42.4 ms, total: 254 ms
Wall time: 2min 3s


In [13]:
%%time
!python run_baseline.py --dataset diabetes --exp_name baselines --note run1

RMSE score
  Scenario  miss_pct [0, 1]  ...             median      most_frequent
0      MAR              0.3  ...  0.1868 +/- 0.0033  0.2231 +/- 0.0115

[1 rows x 9 columns]


Wasserstein score
  Scenario  miss_pct [0, 1]  ...             median      most_frequent
0      MAR              0.3  ...  0.1756 +/- 0.0094  0.2112 +/- 0.0053

[1 rows x 9 columns]


AUROC score
  Scenario  miss_pct [0, 1]  ...       median most_frequent
0      MAR              0.3  ...  0.0 +/- 0.0   0.0 +/- 0.0

[1 rows x 9 columns]
CPU times: user 215 ms, sys: 31.4 ms, total: 247 ms
Wall time: 1min 59s


In [14]:
%%time
!python run_baseline.py --dataset climate --exp_name baselines --note run1

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

In [15]:
%%time
!python run_baseline.py --dataset credit --exp_name baselines --note run1

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

In [16]:
%%time
!python run_baseline.py --dataset raisin --exp_name baselines --note run1

RMSE score
  Scenario  miss_pct [0, 1]  ...             median    most_frequent
0      MAR              0.3  ...  0.2103 +/- 0.0055  0.55 +/- 0.0326

[1 rows x 9 columns]


Wasserstein score
  Scenario  miss_pct [0, 1]  ...             median      most_frequent
0      MAR              0.3  ...  0.1873 +/- 0.0054  0.6041 +/- 0.0513

[1 rows x 9 columns]


AUROC score
  Scenario  miss_pct [0, 1]  ...             median      most_frequent
0      MAR              0.3  ...  0.9951 +/- 0.0054  0.9851 +/- 0.0213

[1 rows x 9 columns]
CPU times: user 2.32 s, sys: 334 ms, total: 2.66 s
Wall time: 21min 38s


In [17]:
%%time
!python run_baseline.py --dataset compression --exp_name baselines --note run1

RMSE score
  Scenario  miss_pct [0, 1]  ...             median    most_frequent
0      MAR              0.3  ...  0.2299 +/- 0.0339  0.36 +/- 0.0304

[1 rows x 9 columns]


Wasserstein score
  Scenario  miss_pct [0, 1]  ...             median      most_frequent
0      MAR              0.3  ...  0.2147 +/- 0.0164  0.3437 +/- 0.0456

[1 rows x 9 columns]


AUROC score
  Scenario  miss_pct [0, 1]  ...       median most_frequent
0      MAR              0.3  ...  0.0 +/- 0.0   0.0 +/- 0.0

[1 rows x 9 columns]
CPU times: user 2.47 s, sys: 400 ms, total: 2.87 s
Wall time: 23min 27s


In [18]:
%%time
!python run_baseline.py --dataset wine --exp_name baselines --note run1

RMSE score
  Scenario  miss_pct [0, 1]  ...            median     most_frequent
0      MAR              0.3  ...  0.173 +/- 0.0202  0.2461 +/- 0.064

[1 rows x 9 columns]


Wasserstein score
  Scenario  miss_pct [0, 1]  ...             median      most_frequent
0      MAR              0.3  ...  0.1507 +/- 0.0178  0.1979 +/- 0.0431

[1 rows x 9 columns]


AUROC score
  Scenario  miss_pct [0, 1]  ...            median      most_frequent
0      MAR              0.3  ...  0.7918 +/- 0.006  0.7901 +/- 0.0085

[1 rows x 9 columns]
CPU times: user 2.01 s, sys: 301 ms, total: 2.31 s
Wall time: 18min 58s


In [19]:
%%time
!python run_baseline.py --dataset obesity --exp_name baselines --note run1

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi